# Simplified Hidden Markov Model



In [1]:
from collections import Counter
from typing import List, Tuple
import math

In [2]:
class SimpleHMM:
    def __init__(self):
        self.transition_counts = Counter()
        self.emission_counts = Counter()
        self.tag_counts = Counter()
        self.tags = set()
        self.vocab = set()
        self.transition_prob = {}
        self.emission_prob = {}

    def train(self, tagged_sentences: List[List[Tuple[str, str]]]):
        for sentence in tagged_sentences:
            prev_tag = '<START>'
            self.tags.add(prev_tag)

            for word, tag in sentence:
                word = word.lower()
                self.vocab.add(word)
                self.tags.add(tag)

                self.transition_counts[(prev_tag, tag)] += 1
                self.emission_counts[(tag, word)] += 1
                self.tag_counts[tag] += 1

                prev_tag = tag

            self.transition_counts[(prev_tag, '<END>')] += 1
            self.tags.add('<END>')

        self._compute_probabilities()

    def _compute_probabilities(self):
        for (prev_tag, next_tag), count in self.transition_counts.items():
            total_from_prev = sum(
                value
                for (source, _), value in self.transition_counts.items()
                if source == prev_tag
            )
            self.transition_prob[(prev_tag, next_tag)] = count / total_from_prev

        for (tag, word), count in self.emission_counts.items():
            total_for_tag = self.tag_counts[tag]
            self.emission_prob[(tag, word)] = count / total_for_tag

    def viterbi(self, sentence: List[str]) -> List[str]:
        words = [word.lower() for word in sentence]
        tag_list = sorted(tag for tag in self.tags if tag not in {'<START>', '<END>'})

        scores = []
        backpointers = []

        first_scores = {}
        first_backpointers = {}
        for tag in tag_list:
            transition = self.transition_prob.get(('<START>', tag), 0.0)
            emission = self.emission_prob.get((tag, words[0]), 1e-8)
            first_scores[tag] = math.log(transition + 1e-12) + math.log(emission + 1e-12)
            first_backpointers[tag] = None

        scores.append(first_scores)
        backpointers.append(first_backpointers)

        for i in range(1, len(words)):
            current_scores = {}
            current_backpointers = {}

            for tag in tag_list:
                emission = self.emission_prob.get((tag, words[i]), 1e-8)
                best_score = -float('inf')
                best_prev_tag = None

                for prev_tag in tag_list:
                    transition = self.transition_prob.get((prev_tag, tag), 1e-8)
                    score = scores[i - 1][prev_tag] + math.log(transition + 1e-12) + math.log(emission + 1e-12)

                    if score > best_score:
                        best_score = score
                        best_prev_tag = prev_tag

                current_scores[tag] = best_score
                current_backpointers[tag] = best_prev_tag

            scores.append(current_scores)
            backpointers.append(current_backpointers)

        best_last_tag = None
        best_last_score = -float('inf')
        for tag in tag_list:
            end_transition = self.transition_prob.get((tag, '<END>'), 1e-8)
            score = scores[-1][tag] + math.log(end_transition + 1e-12)
            if score > best_last_score:
                best_last_score = score
                best_last_tag = tag

        best_tags = [best_last_tag]
        for i in range(len(words) - 1, 0, -1):
            best_tags.append(backpointers[i][best_tags[-1]])

        best_tags.reverse()
        return best_tags

    def predict(self, sentence: List[str]) -> List[Tuple[str, str]]:
        tags = self.viterbi(sentence)
        return list(zip(sentence, tags))

In [3]:
train_data = [
    [('the', 'DET'), ('dog', 'NOUN'), ('barks', 'VERB')],
    [('a', 'DET'), ('cat', 'NOUN'), ('sleeps', 'VERB')],
    [('the', 'DET'), ('cat', 'NOUN'), ('meows', 'VERB')],
    [('dogs', 'NOUN'), ('run', 'VERB')],
]

model = SimpleHMM()
model.train(train_data)

test_sentence = ['the', 'cat', 'barks']
prediction = model.predict(test_sentence)
prediction

[('the', 'DET'), ('cat', 'NOUN'), ('barks', 'VERB')]

## Notes

This simplified version removes:

- smoothing details beyond small fallback probabilities
- forward-backward inference
- visualization code
- graph rendering and probability heatmaps

It is intended as a teaching version before moving to the more complete notebook.